In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import BatchSampler, DataLoader, Dataset, RandomSampler, SequentialSampler
import json

Download the 25lines_100drugs_pca_data.zip file from this google drive folder and upload into colab before running the next line which will extract the contents into the working area

In [6]:
!unzip 25lines_100drugs_pca_data_v2.zip

Archive:  25lines_100drugs_pca_data_v2.zip
   creating: content/
   creating: content/data/
   creating: content/data/Tahoe/
   creating: content/data/Tahoe/subsets/
   creating: content/data/Tahoe/subsets/cfg_25lines_100drugs/
   creating: content/data/Tahoe/subsets/cfg_25lines_100drugs/cells/
   creating: content/data/Tahoe/subsets/cfg_25lines_100drugs/checkpoints/
  inflating: content/data/Tahoe/subsets/cfg_25lines_100drugs/checkpoints/selection_checkpoint.zip  
  inflating: content/data/Tahoe/subsets/cfg_25lines_100drugs/coverage_report.json  
  inflating: content/data/Tahoe/subsets/cfg_25lines_100drugs/download_progress.json  
  inflating: content/data/Tahoe/subsets/cfg_25lines_100drugs/download_status.json  
  inflating: content/data/Tahoe/subsets/cfg_25lines_100drugs/expression_files.json  
   creating: content/data/Tahoe/subsets/cfg_25lines_100drugs/metadata/
  inflating: content/data/Tahoe/subsets/cfg_25lines_100drugs/panel_review.json  
  inflating: content/data/Tahoe/subsets

In [7]:
SEED = 42
PCA_DIR = "/content/content/data/Tahoe/subsets/cfg_25lines_100drugs/splits/pair_seed42_val20_raw4/pca50_hvg2000"

In [18]:
def make_hvg_col(hvg_ids):
    hvg_col = np.full(int(hvg_ids.max()) + 1, -1, dtype=np.int64)
    hvg_col[hvg_ids] = np.arange(len(hvg_ids))
    return hvg_col

def load_pca_model(pca_dir=PCA_DIR):
    with np.load(Path(pca_dir) / "pca_model.npz") as saved:
        model = {key: saved[key] for key in saved.files}
    model["pc_scale"] = float(model["pc_scale"])
    model["hvg_col"] = make_hvg_col(model["hvg_ids"])
    return model


class TahoePCs(Dataset):
    """Self-contained: only needs the files under `pca_dir` (pca_model.npz, pcs_*.npy, cluster_*.npy,
    line_id_*.npy, drug_id_*.npy, dose_*.npy, pca_manifest.json). Copy that folder anywhere and this class works
    with no raw Parquet loader and no Drive mount."""

    def __init__(self, split, pca_dir=PCA_DIR, pc_scale=None):
        pca_dir = Path(pca_dir)
        labels = json.loads((pca_dir / "pca_manifest.json").read_text())
        pcs = np.load(pca_dir / f"pcs_{split}.npy")
        keep = np.isfinite(pcs).all(axis=1)      # all rows unless this was a MAX_BATCHES smoke run
        self.pc_scale = float(np.load(pca_dir / "pca_model.npz")["pc_scale"]) if pc_scale is None else pc_scale
        self.pcs = torch.from_numpy(pcs[keep] / self.pc_scale).float()
        self.cluster = torch.from_numpy(np.load(pca_dir / f"cluster_{split}.npy")[keep].astype(np.int64))
        self.drug_id = np.load(pca_dir / f"drug_id_{split}.npy")[keep].astype(np.int64)
        self.line_id = np.load(pca_dir / f"line_id_{split}.npy")[keep].astype(np.int64)
        self.dose = torch.from_numpy(np.load(pca_dir / f"dose_{split}.npy")[keep].astype(np.float32))  # dose in uM
        self.drug_names = np.asarray(labels["drug_names"])
        self.line_names = np.asarray(labels["line_names"])
        self.is_control = torch.from_numpy(self.drug_id == 0)

    def __len__(self):
        return len(self.pcs)

    def __getitem__(self, index):
        index = np.asarray(index)
        return {"pcs": self.pcs[index], "drug": self.drug_names[self.drug_id[index]].tolist(),
                "cell_line_id": self.line_names[self.line_id[index]].tolist(),
                "dose": self.dose[index],
                "cluster": self.cluster[index], "is_control": self.is_control[index]}


def make_pca_loaders(batch_size=256, seed=SEED, pca_dir=PCA_DIR):
    train_ds = TahoePCs("train", pca_dir)
    val_ds = TahoePCs("validation", pca_dir, pc_scale=train_ds.pc_scale)
    generator = torch.Generator().manual_seed(seed)
    train = DataLoader(train_ds, batch_size=None, num_workers=0,
                       sampler=BatchSampler(RandomSampler(train_ds, generator=generator), batch_size, drop_last=True))
    validation = DataLoader(val_ds, batch_size=None, num_workers=0,
                            sampler=BatchSampler(SequentialSampler(val_ds), batch_size, drop_last=False))
    return train, validation


def pcs_to_hvg_expression(pcs, model=None):
    """Scaled PCs (as served by the loaders) -> log-normalised HVG expression, shape (..., N_HVG)."""
    model = load_pca_model() if model is None else model
    z = (np.asarray(pcs, dtype=np.float64) * model["pc_scale"]) @ model["components"] + model["pca_mean"]
    return z * model["scaler_std"] + model["scaler_mean"]


pca_train_loader, pca_validation_loader = make_pca_loaders()
pca_model = load_pca_model()
print("PC training cells:", len(pca_train_loader.dataset), "| PC validation cells:", len(pca_validation_loader.dataset))
batch = next(iter(pca_train_loader))
print({key: (tuple(v.shape) if torch.is_tensor(v) else f"list[{len(v)}]") for key, v in batch.items()})
print("first cell:", batch["drug"][0], batch["cell_line_id"][0], "dose", float(batch["dose"][0]), "cluster", int(batch["cluster"][0]))

idx = 100  # Just a random index to grab from the batch

datapoint = {
    "drug": batch["drug"][idx],
    "dose": batch["dose"][idx],
    "cell_line_id": batch["cell_line_id"][idx],
    "cluster": batch["cluster"][idx].item(),
    "is_control": batch["is_control"][idx].item(),
    "pcs": batch["pcs"][idx],  # Tensor of principal components
}

print("\nExample Datapoint:")
for key, val in datapoint.items():
    print(f"{key}: {val}")

PC training cells: 791249 | PC validation cells: 200650
{'pcs': (256, 50), 'drug': 'list[256]', 'cell_line_id': 'list[256]', 'dose': (256,), 'cluster': (256,), 'is_control': (256,)}
first cell: DMSO_TF CVCL_C466 dose 0.0 cluster -1

Example Datapoint:
drug: Vemurafenib
dose: 0.05000000074505806
cell_line_id: CVCL_0218
cluster: 2
is_control: False
pcs: tensor([-3.9051e+00,  1.2748e+00,  1.3292e-01, -8.8130e-01, -3.6209e+00,
        -3.7104e-01,  2.9337e+00,  2.0078e+00,  4.3946e+00, -3.0938e+00,
         5.7254e-01,  5.1877e-01, -1.0248e+00,  1.8316e-01, -1.9366e-01,
        -4.7367e-01,  5.8366e-02,  3.8626e-01,  6.6601e-02, -1.0340e-01,
        -1.1184e-01, -2.6131e-01, -1.9633e-03, -6.7569e-01,  2.3259e-01,
        -7.5682e-01,  7.3458e-01, -1.7159e-01, -5.5431e-01, -3.0024e-01,
         7.4073e-01, -7.2089e-01,  5.9451e-01,  1.1933e+00,  2.5357e-01,
        -1.2048e-01,  5.6293e-01, -5.0016e-01,  3.1482e-01, -5.5310e-02,
        -2.0001e-01,  4.2983e-01,  6.0512e-01,  1.5777e+00, -8